# Epstein Files
The documents have been sources from a [GitHub repo](https://github.com/epstein-docs/epstein-docs.github.io/tree/main#) that has OCR'd documents using OpenAI.

The repo was last updated 4 months ago, so the files are not recent. This is something I will work on soon.

## Converting Into Parquet By Doc Types

In [22]:
import glob
import json
from collections import defaultdict
import pandas as pd
import duckdb

In [12]:
all_files = glob.glob("./docs/**/*.json")

# Dictionary of [doc type] -> [file names]
doc_types = defaultdict(list[str])

for file_name in all_files:
    with open(file_name, "rt", encoding="utf-8") as file:
        data = json.load(file)

        metadata = data["document_metadata"]

        dt = metadata["document_type"].lower() if metadata["document_type"] is not None else "unknown"

        doc_types[dt].append(file_name)

In [43]:
# Getting document frequency and writing to csv for Tableu visuals

df = pd.DataFrame({
    "doc_type": [k for k in doc_types.keys()],
    "file_names": [v for v in doc_types.values()],
    "frequency": [len(v) for v in doc_types.values()]
})
df.to_csv("document_frequencies.csv", index=False)

In [ ]:
# These document types are too specific, i.e. 'court document' vs 'court transcript' vs 'court order'
target = "log"
targets = ["court", "email", "report", "myspace", "log"]

query = f"""
    WITH
        combined AS (
            SELECT
                *
            FROM
                df
            WHERE
                doc_type ILIKE '%{target}%'
        )
    SELECT
        '{target}_documents' AS doc_type,
        list_concat(list(file_names)) AS file_names,
        SUM(frequency) AS frequency
    FROM
        combined
"""

query = f"""
    SELECT
        doc_type,
        frequency
    FROM
        df
    WHERE
        regexp_matches(doc_type, '(?i)\\b{target}\\b')
"""
duckdb.query(query)

┌───────────────────────────────┬───────────┐
│           doc_type            │ frequency │
│            varchar            │   int64   │
├───────────────────────────────┼───────────┤
│ log entries by location       │         6 │
│ log entries by event date     │       120 │
│ suicide watch observation log │        31 │
│ telecommunications log        │         1 │
│ inmate observation log        │        15 │
│ email or message log          │         3 │
│ communication log             │         2 │
│ log entries                   │         7 │
│ daily lieutenant's log        │        30 │
│ log or record                 │        27 │
│     ·                         │         · │
│     ·                         │         · │
│     ·                         │         · │
│ log sheet                     │         2 │
│ log or visitation record      │         1 │
│ inmate record or log          │         1 │
│ shift log or roster           │         1 │
│ pilot's flight log            │ 